# Trading Bot – Smoke Test (Schnelltest)

**Zweck:** Vollständiger End-to-End-Test aller Pipeline-Schritte in ~10–15 Minuten.

**Was reduziert ist:**
- **Assets:** 15 (statt ~260)
- **Walk-Forward-Folds:** ~2 (statt 12, `train_years=14`)
- **Epochen pro Fold:** 3 (statt 50, kein Early-Stopping)

**Was NICHT reduziert ist (alle Code-Pfade werden durchlaufen):**
- Alle Pipeline-Schritte (Clone, Install, Daten, Training, Backtest, Chart, Manifest, Upload)
- Equity-Chart (`v2_7d_equity.png`)
- Run-Manifest (`run_manifest.json`)
- Archiv (`kaggle_artifacts.tar.gz` mit Checkpoints)
- Dataset-Upload (falls KAGGLE_KEY gesetzt)

**Anleitung:**
1. Accelerator auf **GPU T4 x2** stellen (Settings → Accelerator)
2. Dataset **busersteven/trading-raw-data** hinzufügen (Add data → Your datasets)
3. **Run All** – fertig in ca. 10–15 Min
4. Wenn alles grün: das echte Notebook `kaggle_notebook.ipynb` starten

In [ ]:
# Smoke-Test-Pipeline: alle Schritte, stark verkleinert
import os
import re
import subprocess
import sys
import time

# ── Smoke-Test aktivieren ─────────────────────────────────────────────────────
# Setzt alle Reduktionen: 15 Assets, 3 Epochen, ~2 Folds
os.environ["KAGGLE_SMOKE_TEST"]   = "1"

# Nur Horizont 7d (schnellster sinnvoller Test)
HORIZONS = [7]
os.environ["KAGGLE_SH_HORIZONS"] = ",".join(str(h) for h in HORIZONS)

# TorchDynamo-Fix (wie im echten Run)
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

print(f"KAGGLE_SMOKE_TEST       = {os.environ['KAGGLE_SMOKE_TEST']}")
print(f"KAGGLE_SH_HORIZONS      = {os.environ['KAGGLE_SH_HORIZONS']}")
print(f"TORCHDYNAMO_DISABLE     = {os.environ['TORCHDYNAMO_DISABLE']}")
print()
print("Erwartete Laufzeit: ~10-15 Minuten")
print("Alle Pipeline-Schritte werden durchlaufen (nur Datenvolumen reduziert)")

# ── Gleichen Pipeline-Code wie echter Run laden ───────────────────────────────
cache_bust = int(time.time())
url = (
    f"https://raw.githubusercontent.com/stevenlangeshops/trading/main/"
    f"scripts/kaggle_full_run.py?cb={cache_bust}"
)
r = subprocess.run(
    ["wget", "-q", "-O", "/kaggle/working/kaggle_full_run.py", url],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(r.stdout or "download ok")

path = "/kaggle/working/kaggle_full_run.py"
with open(path, encoding="utf-8") as f:
    code = f.read()

# Fallback fuer aeltere Script-Versionen mit fester SH_HORIZONS-Zeile
code, n_legacy = re.subn(
    r"^(\s*SH_HORIZONS\s*=\s*)\[\s*\d+\s*,\s*\d+\s*\]",
    lambda m: m.group(1) + repr(HORIZONS),
    code,
    flags=re.MULTILINE,
)
if n_legacy:
    print(f"[legacy] {n_legacy} SH_HORIZONS-Zeile(n) auf {HORIZONS!r} gesetzt")

# Modul-Cache leeren
for mod in list(sys.modules.keys()):
    if mod.startswith(("strategy", "models", "features", "config_v2", "train_v2", "backtest_v2")):
        del sys.modules[mod]

exec(code)